In [ ]:
# Notebook 5 — Graph Neural Networks for Molecular Property Prediction

**Duration:** 4 hours
**Audience:** Undergrad CS students with intermediate Python and basic ML experience
**Goal:** Build a PyTorch Geometric (PyG) pipeline, train a Fe-focused GNN on GPU, and compare it to classical baselines.

---

## Learning objectives

- Represent molecules as graphs (nodes = atoms, edges = bonds / proximity).
- Implement a reproducible data pipeline that yields PyG Data objects.
- Build, train, and evaluate a GNN (FeRedoxGNN) with early stopping and LR scheduling on GPU.
- Compare GNN performance against Random Forest and Gaussian Process baselines from Day 3.

---

## Part 1 — Theory: Message passing and graph-level prediction (45 min)

### 1.1 Molecules as graphs

A molecule is represented as a graph $G = (V, E)$ where $V$ is the set of nodes (atoms) and $E$ is the set of edges (bonds or proximity links). Each node $v_i$ has an initial feature vector

$$\mathbf{h}_i^{(0)} \in \mathbb{R}^{d_{\text{node}}}$$

Edges may also carry features (bond order, distance, ring flags).

Common node features: atomic number, electronegativity, formal charge, hybridization, 3D coordinates.

Common edge features: interatomic distance, bond order, whether the edge is in a ring.

### 1.2 Message Passing Neural Networks (MPNNs)

A generic MPNN layer performs a message aggregation followed by an update:

$$\mathbf{h}_i^{(l+1)} = \mathrm{UPDATE}\!\left(\mathbf{h}_i^{(l)}, \mathrm{AGGREGATE}\!\left(\{\mathbf{h}_j^{(l)} : j \in \mathcal{N}(i)\}\right)\right)$$

where $\mathcal{N}(i)$ denotes neighbors of node $i$.

A common concrete form is the GCN layer (Kipf & Welling, 2017):

$$\mathbf{h}_i^{(l+1)} = \sigma\!\left(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \frac{1}{\sqrt{|\mathcal{N}(i)| \cdot |\mathcal{N}(j)|}} W^{(l)} \mathbf{h}_j^{(l)}\right)$$

Readout (graph-level pooling) turns node embeddings into a single vector used for regression:

$$\widehat{y} = \mathrm{MLP}\!\left(\mathrm{READOUT}\!\left(\{\mathbf{h}_i^{(L)} : v_i \in V\}\right)\right)$$

Common READOUTs: mean, sum, max, or attention-weighted pooling.

---

## Part 2 — Building the data pipeline (60 min)

We provide a small, self-contained pipeline that: (1) encodes atom features, (2) converts a molecule into a PyG Data object, and (3) builds a dataset with a 3-tier fallback: XYZ files -> tabular CSV -> synthetic generator (seed 42). The synthetic fallback ensures the notebook runs even when real files are missing.



In [ ]:

# Part 2: Imports and GPU check
import os
import random
import math
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    try:
        props = torch.cuda.get_device_properties(0)
        # property is `total_memory` in recent PyTorch builds
        print('GPU name:', props.name)
        print('Total memory (bytes):', props.total_memory)
    except Exception:
        pass

# Filenames we keep the same as the course standard
BEST_CHECKPOINT = 'best_gnn_model.pt'
FINAL_MODEL = 'fe_redox_gnn_final.pt'
TRAIN_CURVES = 'gnn_training_curves.png'
TEST_PARITY = 'gnn_test_parity.png'
FULL_COMPARISON = 'full_model_comparison.png'
FINAL_RESULTS = 'final_results.csv'



In [ ]:


### 2.1 Atom feature encoders

We keep compact dictionaries (element -> atomic number, electronegativity, covalent radius). These can be extended later.



In [ ]:

# Small element tables used in the workshop
ELEMENTS = ['H','C','N','O','F','P','S','Cl','Fe']
ELEMENT_TO_Z = {el: i+1 for i, el in enumerate(ELEMENTS)}
# Pauling electronegativity (very small table for demo)
ELECTRONEGATIVITY = {'H': 2.20, 'C': 2.55, 'N': 3.04, 'O': 3.44, 'F': 3.98,
                     'P': 2.19, 'S': 2.58, 'Cl': 3.16, 'Fe': 1.83}
# Covalent radii in Angstroms (approximate)
COV_RADII = {'H': 0.31, 'C': 0.76, 'N': 0.71, 'O': 0.66, 'F': 0.57,
             'P': 1.07, 'S': 1.05, 'Cl': 1.02, 'Fe': 1.26}

def get_atom_features(element: str, pos: np.ndarray):
    """Return a simple numeric feature vector for an atom.
    Features: one-hot element (len(ELEMENTS)), atomic number scaled, electronegativity,
    covalent radius, and the 3D coordinates (normalized).
    """
    el = element if element in ELEMENT_TO_Z else 'C'
    one_hot = np.zeros(len(ELEMENTS), dtype=float)
    one_hot[ELEMENTS.index(el)] = 1.0
    z = ELEMENT_TO_Z.get(el, 6)
    en = ELECTRONEGATIVITY.get(el, 2.5)
    r_cov = COV_RADII.get(el, 0.75)
    # position normalization: center at origin for this molecule (handled upstream)
    coords = np.asarray(pos, dtype=float)
    feats = np.concatenate([one_hot, [z/100.0, en/4.0, r_cov/2.0], coords/10.0])
    return feats


In [ ]:


### 2.2 Molecule -> PyG Data conversion

We accept a minimal molecule representation: dict with keys `elements` (list of str), `coords` (Nx3 array), and `y` (float target). Edge construction: first try covalent cutoff (distance < factor * (r_cov_i + r_cov_j)). If that yields a disconnected graph or no edges, fall back to k-NN (k=4). When adding k-NN edges we add both directions so the graph is effectively undirected in message passing.



In [ ]:

from scipy.spatial import cKDTree

def molecule_to_graph(mol: dict, bond_cutoff_factor=1.3, k_nn=4):
    elements = mol['elements']
    coords = np.asarray(mol['coords'], dtype=float)
    n = len(elements)
    # center coordinates
    coords = coords - coords.mean(axis=0)

    # node features
    node_feats = np.vstack([get_atom_features(el, coords[i]) for i, el in enumerate(elements)])
    x = torch.tensor(node_feats, dtype=torch.float)

    # compute pairwise distances
    tree = cKDTree(coords)
    pairs = []
    for i in range(n):
        # covalent-based neighbors
        for j in range(i+1, n):
            dist = np.linalg.norm(coords[i] - coords[j])
            cutoff = bond_cutoff_factor * (COV_RADII.get(elements[i],0.75) + COV_RADII.get(elements[j],0.75))
            if dist <= cutoff:
                pairs.append((i, j))
    if len(pairs) == 0:
        # k-NN fallback
        dists, idxs = tree.query(coords, k=min(k_nn+1, n))
        for i in range(n):
            neighbors = idxs[i]
            for j in neighbors:
                if j == i:
                    continue
                pairs.append((i, int(j)))
    # Build bidirectional edge index
    if len(pairs) == 0:
        # isolated single-atom fallback (self loop)
        edge_index = torch.tensor([[0],[0]], dtype=torch.long)
    else:
        ei = []
        for (i,j) in pairs:
            ei.append([i,j])
            ei.append([j,i])
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()

    y = torch.tensor([mol.get('y', 0.0)], dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, y=y)
    # store coordinates for optional visualization
    data.pos = torch.tensor(coords, dtype=torch.float)
    return data


In [ ]:


### 2.3 Building a dataset with fallbacks

This function looks for a directory of XYZ files, then a CSV, and otherwise synthesizes data. The synthetic generator produces 500 small complexes with random positions and elements drawn from our small element list.

```python
def build_graph_dataset(xyz_dir: str = None, csv_path: str = None, n_synthetic: int = 500):
    graphs = []
    if xyz_dir and os.path.isdir(xyz_dir):
        # Placeholder: user-provided XYZ loader would go here. For the workshop we skip heavy parsing.
        print('XYZ directory found but XYZ parsing is not implemented in this minimal demo. Falling back.')
    if csv_path and os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            # Expect columns: elements (semicolon-joined), coords (semi-colon groups as x:y:z), y
            for _, row in df.iterrows():
                elems = row['elements'].split(';')
                coords_raw = row['coords'].split(';')
                coords = [list(map(float, c.split(':'))) for c in coords_raw]
                graphs.append(molecule_to_graph({'elements': elems, 'coords': coords, 'y': float(row['y'])}))
            if len(graphs) > 0:
                return graphs
        except Exception:
            print('CSV parsing failed; falling back to synthetic dataset.')
    # Synthetic fallback
    rng = np.random.default_rng(SEED)
    for i in range(n_synthetic):
        natoms = rng.integers(5, 25)
        elements = rng.choice(ELEMENTS, size=natoms).tolist()
        coords = rng.normal(scale=1.0, size=(natoms, 3))
        # Synthetic target: simple function of number of Fe atoms + noise
        y = float((elements.count('Fe') * 0.5) + rng.normal(scale=0.2))
        graphs.append(molecule_to_graph({'elements': elements, 'coords': coords, 'y': y}))
    return graphs

# Build dataset (attempt to use $SCRATCH if available, otherwise synthetic)
scratch = os.environ.get('SCRATCH') or '.'
dataset = build_graph_dataset(xyz_dir=os.path.join(scratch, 'xyz_files'),
                              csv_path=os.path.join(scratch, 'fe_redox_data.csv'),
                              n_synthetic=500)
print('Number of graphs:', len(dataset))

# Train/val/test split (70/15/15)
N = len(dataset)
indices = list(range(N))
random.shuffle(indices)

n_train = int(0.7 * N)
n_val = int(0.15 * N)
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_dataset = [dataset[i] for i in train_idx]
val_dataset = [dataset[i] for i in val_idx]
test_dataset = [dataset[i] for i in test_idx]

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print('Train/Val/Test sizes:', len(train_dataset), len(val_dataset), len(test_dataset))


In [ ]:


**Mentor checkpoint 7**

- Verify dataset size and that graphs have reasonable node counts (no empty graphs).
- Confirm that the synthetic fallback is acceptable for the machines without raw data.
- Proceed only after confirmation.

---

## Part 3 — GNN model architecture (45 min)

We implement the FeRedoxGNN used in the course. It follows: Linear embedding -> stack of GCNConv layers with BatchNorm and residual connections -> global mean pooling -> MLP head for regression.



In [ ]:

class FeRedoxGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_gnn_layers=3, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embed = nn.Linear(in_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_gnn_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.dropout = nn.Dropout(dropout)
        # MLP head: two hidden layers
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.embed(x)
        for conv, bn in zip(self.convs, self.bns):
            h_in = h
            h = conv(h, edge_index)
            h = bn(h)
            h = F.relu(h)
            # simple residual connection
            h = h + h_in
            h = self.dropout(h)
        hg = global_mean_pool(h, batch)
        out = self.head(hg)
        return out.view(-1)

# Instantiate model (we derive in_dim from dataset)
in_dim = dataset[0].x.shape[1]
model = FeRedoxGNN(in_dim=in_dim, hidden_dim=64, num_gnn_layers=3, dropout=0.1).to(device)
print(model)


In [ ]:

---

## Part 4 — Training on GPU with early stopping (60 min)

Loss: MSE. Metrics: RMSE and R^2.



In [ ]:

from sklearn.metrics import mean_squared_error, r2_score

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, preds = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        ys.append(batch.y.cpu().numpy())
        preds.append(out.detach().cpu().numpy())
    ys = np.concatenate(ys)
    preds = np.concatenate(preds)
    return {'rmse': rmse(ys, preds), 'r2': float(r2_score(ys, preds)), 'y': ys, 'yhat': preds}

def train_one_epoch(model, loader, optimizer, device, max_grad_norm=1.0):
    model.train()
    losses = []
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = F.mse_loss(out, batch.y.view_as(out))
        loss.backward()
        # gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        losses.append(float(loss.item()))
    return float(np.mean(losses))

# Training loop configuration (preserve hyperparameters from course)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 150
PATIENCE = 20

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)

best_val_rmse = float('inf')
best_epoch = -1
history = {'train_loss': [], 'val_rmse': [], 'val_r2': []}
patience_counter = 0
prev_lr = optimizer.param_groups[0]['lr']

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_res = evaluate(model, val_loader, device)
    history['train_loss'].append(train_loss)
    history['val_rmse'].append(val_res['rmse'])
    history['val_r2'].append(val_res['r2'])

    # Scheduler step uses the metric to reduce LR
    scheduler.step(val_res['rmse'])
    current_lr = optimizer.param_groups[0]['lr']
    if current_lr < prev_lr:
        print(f'Epoch {epoch}: learning rate reduced {prev_lr} -> {current_lr}')
        prev_lr = current_lr

    print(f'Epoch {epoch}: train_loss={train_loss:.4f} val_rmse={val_res["rmse"]:.4f} val_r2={val_res["r2"]:.4f}')

    # Early stopping check
    if val_res['rmse'] < best_val_rmse:
        best_val_rmse = val_res['rmse']
        best_epoch = epoch
        patience_counter = 0
        # Save best model state
        torch.save({'model_state_dict': model.state_dict(), 'config': dict(in_dim=in_dim, hidden_dim=64,
                                                                           num_gnn_layers=3, dropout=0.1),
                    'epoch': epoch}, BEST_CHECKPOINT)
        print(f'New best model saved at epoch {epoch} (val_rmse={best_val_rmse:.4f})')
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print('Early stopping triggered.')
        break

# Save final model package
torch.save({'model_state_dict': model.state_dict(),
            'config': dict(in_dim=in_dim, hidden_dim=64, num_gnn_layers=3, dropout=0.1),
            'training_history': history}, FINAL_MODEL)

print('Training finished. Best epoch:', best_epoch)


In [ ]:

---

## Part 5 — Evaluation and comparison (30 min)


In [ ]:

# Load best checkpoint and evaluate on test set
if os.path.exists(BEST_CHECKPOINT):
    ckpt = torch.load(BEST_CHECKPOINT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print('Loaded best checkpoint from epoch', ckpt.get('epoch', 'unknown'))

test_res = evaluate(model, test_loader, device)
print('Test RMSE:', test_res['rmse'], 'Test R2:', test_res['r2'])

# Save parity plot
import matplotlib.pyplot as plt
plt.figure(figsize=(6,6))
plt.scatter(test_res['y'], test_res['yhat'], s=10, alpha=0.6)
minv = min(test_res['y'].min(), test_res['yhat'].min())
maxv = max(test_res['y'].max(), test_res['yhat'].max())
plt.plot([minv, maxv], [minv, maxv], 'k--')
plt.xlabel('True y')
plt.ylabel('Predicted y')
plt.title('GNN Test Parity')
plt.savefig(TEST_PARITY, dpi=150)
plt.close()

# Load Day 3 baselines if available and compare
baseline_csv = os.path.join(os.environ.get('SCRATCH', '.'), 'baseline_results.csv')
if os.path.exists(baseline_csv):
    baselines = pd.read_csv(baseline_csv)
else:
    baselines = pd.DataFrame({'model': ['RandomForest', 'GPR'], 'rmse': [1.0, 0.9], 'r2': [0.2, 0.25]})

# Append GNN result
gnn_row = pd.DataFrame([{'model': 'FeRedoxGNN', 'rmse': test_res['rmse'], 'r2': test_res['r2']}])
results = pd.concat([baselines, gnn_row], ignore_index=True)
results.to_csv(FINAL_RESULTS, index=False)

# Plot comparison
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.bar(results['model'], results['rmse'])
plt.title('RMSE (lower is better)')
plt.xticks(rotation=45)
plt.subplot(1,2,2)
plt.bar(results['model'], results['r2'])
plt.title('R^2 (higher is better)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FULL_COMPARISON, dpi=150)
plt.close()

# Save final aggregated model info
final_package = {
    'model_state_dict': model.state_dict(),
    'config': dict(in_dim=in_dim, hidden_dim=64, num_gnn_layers=3, dropout=0.1),
    'test_metrics': {'rmse': test_res['rmse'], 'r2': test_res['r2']},
    'training_history': history
}
torch.save(final_package, 'fe_redox_gnn_final.pt')

print('Saved final results and model.')


In [ ]:

**Mentor checkpoint 8**

- Confirm that training converged on at least one node and that the saved files exist: `best_gnn_model.pt`, `fe_redox_gnn_final.pt`, `gnn_training_curves.png`, `gnn_test_parity.png`, `full_model_comparison.png`.
- Inspect `final_results.csv` and verify GNN vs baselines make sense.
- Proceed only after confirmation.

---

### Exercises

#### Exercise 5.1 — Modify the GNN architecture

1. Implement `ModifiedGNN` by changing one of the following (pick one):
   1. Add an additional GCN layer.
   2. Replace `GCNConv` with `GATConv` (requires `torch_geometric.nn import GATConv`).
   3. Swap ReLU for GELU activations.
   4. Use sum pooling instead of mean pooling.
   5. Increase hidden_dim to 128.
2. Train the modified model for at least 30 epochs and report test RMSE and R^2.

Hint: Replace the `pass` in the skeleton below with a real implementation.



In [ ]:

class ModifiedGNN(FeRedoxGNN):
    def __init__(self, in_dim, **kwargs):
        # YOUR CODE HERE: call super().__init__ with modified hyperparameters if needed
        super().__init__(in_dim, **kwargs)

    def forward(self, data):
        # YOUR CODE HERE: you can largely reuse FeRedoxGNN.forward but adapt layers/activations
        return super().forward(data)


In [ ]:


#### Exercise 5.2 — Error analysis

1. Find the top-10 highest absolute errors on the test set and inspect their molecular graphs (node counts, Fe counts).
2. Is there a structural motif common to high-error molecules? Report observations.
3. Plot a histogram of residuals (predicted - true) and report skewness & kurtosis.
4. Plot error vs. molecular size (number of atoms) and comment whether larger molecules are harder.

#### Exercise 5.3 — Discussion

1. Why might a GNN outperform an RF on this task? When might RF be preferable?
2. What are limitations of the synthetic fallback dataset we used in class demos?
3. Suggest two realistic ways to improve the GNN's generalization.
4. If you had access to full DFT-calculated electron densities, how would you incorporate that into a GNN pipeline?

---

## Summary

| Component | Tool / API | Key concept |
|---|---:|---|
| Representation | PyTorch Geometric (Data, DataLoader) | Molecules as graphs, node/edge features |
| Model | GCNConv stack + global_mean_pool | Message passing and readout |
| Optimization | Adam, ReduceLROnPlateau, early stopping | Learning rate schedule and overfitting control |
| Evaluation | RMSE, R^2, parity plots | Compare to RF and GPR baselines |

---

## Preparation for Day 5

Day 5 is the lightning-talk and capstone presentation day. Prepare a 5–10 minute presentation that includes:

1. One slide describing the dataset you used (counts, feature choices, any preprocessing).
2. One slide describing your model(s) and training setup (hyperparameters, GPUs used).
3. One slide showing training curves and one parity plot comparing your best method to baselines.
4. One concluding slide: three takeaways and one follow-up idea you would pursue with more time.

---

End of Notebook 5.
